# 02. Creating an Azure AI Foundry Prompt Agent

This is the section's first real **Azure AI Foundry Agent Service** example. It uses `azure-ai-projects`'
`AIProjectClient` to create a **prompt agent** — a named, versioned agent definition (`PromptAgentDefinition`)
consisting of a model deployment and a system prompt, with no tools attached yet. Running this script
registers the agent (`"IT-HelpDesk-Agent"`) as version 1 inside your Foundry project; later scripts in this
section (`03_invoke_agent.py`, `05_IT_HelpDesk_agent.py`, `07_helpdesk_client.py`) reuse or add new versions
of this exact agent by name.

This maps to the AI-102 exam domain "Implement generative AI solutions" — specifically the Azure AI Foundry
Agent Service's agent lifecycle (define → create/version → invoke → iterate).

**Difficulty:** Beginner


## Prerequisites

**pip3 packages** (already in the repo's root `requirements.txt`):
- `azure-ai-projects==2.2.0`
- `azure-identity==1.25.3`
- `python-dotenv` (added here for env-driven config)

**Azure resource(s) required:**
- An Azure AI Foundry **project** with at least one deployed chat-completion model (this section's course
  example is Foundry project `ai-103-project`).

**Environment variables** (add to the root `.env`, consistent with the pattern already used in
`11_azure_ai_foundry/`):
- `AZURE_AI_PROJECT_ENDPOINT` — your Foundry project endpoint, e.g.
  `https://<your-resource>.services.ai.azure.com/api/projects/<your-project>`. Already present in this repo's
  `.env` (pointed at a different lab project than this course's `ai-103-project`, so it's fine to reuse the
  same variable name here).
- `AZURE_AI_MODEL_DEPLOYMENT` — the name of a deployed chat model in that project (the original script hardcodes
  `"gpt-5.4"`, a placeholder/course-specific deployment name — use whatever deployment name actually exists in
  your project).
- `AZURE_AI_AGENT_NAME` (optional) — defaults to `"IT-HelpDesk-Agent"` if unset.

Auth is Entra ID via `DefaultAzureCredential` — run `az login` first, no API key needed.


## What You'll Learn

- How `AIProjectClient` connects to a Foundry project using Entra ID (`DefaultAzureCredential`) instead of an API key
- What a `PromptAgentDefinition` is: a stateless, model + system-prompt agent definition (no tools yet)
- How `client.agents.create_version(agent_name=..., definition=...)` creates (or *versions*) a named agent — running this twice creates version 2, not a duplicate agent
- Why "agent name" is the stable identity you invoke later, while "version" tracks each definition change


### Setup and client construction

We load the project endpoint and deployment name from environment variables instead of the hardcoded strings
in the original script. `DefaultAzureCredential` tries several auth methods in order (environment variables,
managed identity, Azure CLI, etc.) and in local development typically resolves to your `az login` session.

- **💡 Exam tip:** AI-102 favors **Entra ID / `DefaultAzureCredential`** as the recommended authentication
  pattern for Azure AI services over API keys — no secret to store or rotate, and access is governed by Azure
  RBAC role assignments (e.g. *Azure AI User* on the Foundry project).
- **🔄 Alternatives:** `AzureCliCredential` (used later in `09_customer_rag_agent.py`) authenticates using only
  the Azure CLI's cached login, which is more predictable in local dev than `DefaultAzureCredential`'s broader
  credential chain. For non-interactive/service scenarios, a `ClientSecretCredential` (service principal) or
  managed identity would replace it.


In [ ]:
import os

from dotenv import load_dotenv
from azure.ai.projects import AIProjectClient
from azure.ai.projects.models import PromptAgentDefinition
from azure.identity import DefaultAzureCredential

load_dotenv()

PROJECT_ENDPOINT = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
DEPLOYMENT_NAME = os.getenv("AZURE_AI_MODEL_DEPLOYMENT", "gpt-5.4")
AGENT_NAME = os.getenv("AZURE_AI_AGENT_NAME", "IT-HelpDesk-Agent")

client = AIProjectClient(
    endpoint=PROJECT_ENDPOINT,
    credential=DefaultAzureCredential(),
)


### Creating the agent version

`PromptAgentDefinition(model=..., instructions=...)` is the agent's definition: which model deployment to use
and what its system prompt is. `client.agents.create_version(agent_name=AGENT_NAME, definition=...)` publishes
that definition under the given name. If `AGENT_NAME` doesn't exist yet, this creates version 1; if it already
exists (as it will after `05_IT_HelpDesk_agent.py` runs later in this section), this creates the *next*
version — the agent's history is preserved, not overwritten.

- **💡 Exam tip:** Know the distinction between a **prompt agent** (`PromptAgentDefinition` — stateless,
  invoked per-request, no server-side conversation memory of its own) and a **hosted/persisted agent with
  threads** (seen in `11_azure_ai_foundry/05_hosted_agent`), which maintains multi-turn conversation state
  server-side. Both are "Agent Service" resources, but they solve different problems.
- **🔄 Alternatives:** A `PromptAgentDefinition` with no `tools` (this script) vs. one with `tools=[...]`
  (`FunctionTool`, `WebSearchTool`, `FileSearchTool` — seen in `04_agent_web_search.py` and
  `05_IT_HelpDesk_agent.py`) — tools are what turn a plain chat agent into one that can take actions or fetch
  grounding data.


In [ ]:
agent = client.agents.create_version(
    agent_name=AGENT_NAME,
    definition=PromptAgentDefinition(
        model=DEPLOYMENT_NAME,
        instructions=(
            "You are an IT support assistant for a company. "
            "Help users with password resets, VPN issues, and software installation. "
            "Give clear, step-by-step answers. "
            "If the question is outside IT support topics, politely say so."
        )
    )
)

print("Agent created:")
print(f"  ID      : {agent.id}")
print(f"  Name    : {agent.name}")
print(f"  Version : {agent.version}")


## Summary

Running this notebook (with real credentials) creates — or adds a new version to — a Foundry agent named
`IT-HelpDesk-Agent`, defined by a model deployment and a system prompt, with no tools attached. The printed
output is the agent's Foundry-assigned `id`, its stable `name`, and its numeric `version`. That `name` is what
every downstream script in this section (`03`, `05`, `07`) uses to find and invoke this exact agent again —
Foundry resolves the name to a specific (or latest) version at invocation time.


## Try It Yourself

1. **Easy:** Change the `instructions` string to give the agent a different persona (e.g. an HR assistant)
   and re-run — note this creates version 2 of the *same* agent name rather than a new agent.
2. **Intermediate:** Print `agent` in full (not just `id`/`name`/`version`) to see what other metadata the
   Foundry SDK returns about a freshly created agent version.
3. **Advanced:** Write a small helper that calls `client.agents.list_versions(agent_name=AGENT_NAME)` (check
   the `azure-ai-projects` SDK reference for the exact method) to enumerate every version you've created so
   far, and print each version's `instructions` to see the history.
